# 13. 最佳實踐

學習 LangGraph 開發的最佳實踐和生產模式。

---

## 🎯 學習目標

- ✅ 狀態設計原則
- ✅ 錯誤處理模式
- ✅ 測試策略
- ✅ 效能優化

In [1]:
from typing import TypedDict, Annotated, Optional
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

## 13.1 狀態設計

| 原則 | 說明 |
|------|------|
| 明確類型 | 使用 TypedDict |
| 選擇 Reducer | 累加用 add_messages |
| 可選欄位 | 使用 `| None` |

In [2]:
class GoodState(TypedDict):
    """良好的狀態設計"""
    messages: Annotated[list, add_messages]  # 明確 reducer
    current_step: str                        # 追蹤步驟
    error: str | None                       # 可選錯誤
    metadata: dict                           # 額外資訊

print("✅ 良好的狀態設計範例")

✅ 良好的狀態設計範例


## 13.2 節點設計

In [3]:
def good_node(state: GoodState) -> dict:
    """良好的節點設計
    
    特點：
    1. 類型標註
    2. 文檔說明
    3. 錯誤處理
    4. 日誌輸出
    """
    try:
        print(f"  📍 執行節點...")
        result = "處理結果"
        return {"current_step": "completed", "error": None}
    except Exception as e:
        print(f"  ❌ 錯誤: {e}")
        return {"current_step": "error", "error": str(e)}

print("✅ 節點範例已定義")

✅ 節點範例已定義


## 13.3 測試模式

In [4]:
def test_node_isolation():
    """測試單一節點"""
    state = {"messages": [], "current_step": "start", "error": None, "metadata": {}}
    result = good_node(state)
    assert result["current_step"] == "completed"
    assert result["error"] is None
    print("✅ 節點測試通過")

test_node_isolation()

  📍 執行節點...
✅ 節點測試通過


In [5]:
def test_full_graph():
    """測試完整圖"""
    graph = StateGraph(GoodState)
    graph.add_node("process", good_node)
    graph.add_edge(START, "process")
    graph.add_edge("process", END)
    app = graph.compile()
    
    result = app.invoke({"messages": [], "current_step": "", "error": None, "metadata": {}})
    assert result["current_step"] == "completed"
    print("✅ 完整圖測試通過")

test_full_graph()

  📍 執行節點...
✅ 完整圖測試通過


## 💡 最佳實踐總結

### 設計原則
1. **清晰的狀態定義** - 使用 TypedDict
2. **單一職責節點** - 每個節點做一件事
3. **錯誤處理** - 捕獲並記錄錯誤
4. **可測試性** - 節點可獨立測試

### 生產建議
- 使用 checkpointer 持久化
- 設定重試機制
- 加入監控和日誌

---

🎉 恭喜完成 LangGraph 課程！